# Feature Extraction From Captured Network Packets

This notebook builds the dataset for device classification from network captures.

### Pipeline Steps
1. Read each PCAP with tshark.
2. Map source and destination addresses to device labels.
3. Build sliding-window features (UP, DOWN, TOT).
4. Save one CSV per PCAP subfolder.

### Input Location
- Place all `.pcapng` files under the project root.
- By default, `ROOT_PATH` is `.` (the notebook folder).
- Subfolders are supported, for example `MatterThread` and `Zigbee`.

### Mapping File Rule
- For each capture, place a mapping CSV in the same folder.
- Naming pattern: `<pcap_file_stem>_mapping.csv`.
- Example: `MatterCapture1.pcapng` -> `MatterCapture1_mapping.csv`.

### Output Split
- One output CSV is generated for each subfolder that contains PCAP files.
- Example: `MatterThread/*.pcapng` -> `dataset_MatterThread_5s.csv`.
- Example: `Zigbee/*.pcapng` -> `dataset_Zigbee_5s.csv`.

### Output Location
- Generated files are written under `OUTPUT_DIR`.

### Imports and configuration

In [1]:
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from io import StringIO
from pathlib import Path
import multiprocessing as mp
import os
import subprocess
import threading
import numpy as np
import pandas as pd
try:
    from IPython.display import clear_output
except Exception:
    clear_output = None

# Runtime configuration
TSHARK_PATH = "tshark"
ROOT_PATH = Path(".")
OUTPUT_DIR = ROOT_PATH / "outputs"
WINDOW_SIZE = 5
FILL_NA_VALUE = -1

# Tshark fields and standardized output names
TSHARK_FIELDS = [
    "frame.number",
    "frame.time",
    "frame.len",
    "wpan.src16",
    "wpan.dst16",
    "wpan.src64",
    "wpan.dst64",
    "wpan.seq_no",
    "wpan.fcf",
    "wpan.fcs",
]

COLUMN_RENAME_MAP = {
    "frame.number": "Packet Number",
    "frame.time": "Time",
    "frame.len": "Length",
    "wpan.src16": "Source IEEE",
    "wpan.dst16": "Destination IEEE",
    "wpan.src64": "Source IEEE 64-bit",
    "wpan.dst64": "Destination IEEE 64-bit",
    "wpan.seq_no": "Sequence Number",
    "wpan.fcf": "Frame Control Field",
    "wpan.fcs": "Frame Check Sequence",
}

print(f"Root path: {ROOT_PATH.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Root path: /Users/max/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Articoli/Zigbee vs Mot/Data Article
Output directory: /Users/max/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Articoli/Zigbee vs Mot/Data Article/outputs


### Function definitions

In [2]:
CPU_COUNT = os.cpu_count() or 4


class NotebookProgressTracker:
    def __init__(self, total_steps: int = 4):
        self.total_steps = total_steps
        self._status_by_file: dict[str, str] = {}
        self._lock = threading.Lock()

    def update_step(self, file_name: str, step_idx: int, step_name: str) -> None:
        with self._lock:
            self._status_by_file[file_name] = f"{file_name} - Step {step_idx}/{self.total_steps} {step_name}"
            self._render_locked()

    def complete(self, file_name: str, fail=False) -> None:
        with self._lock:
            if fail:
                self._status_by_file[file_name] = f"{file_name} - Failed"
            else:
                self._status_by_file[file_name] = f"{file_name} - Completed"
            self._render_locked()

    def _render_locked(self) -> None:
        if clear_output is not None:
            clear_output(wait=True)
        for file_name in sorted(self._status_by_file):
            print(self._status_by_file[file_name])


# Allow ProcessPool only when running with fork (safe in notebooks).
def _step3_can_use_process_pool() -> bool:
    if os.name != "posix":
        return False
    return True


# Adapter for executor.map in device-parallel mode.
def _build_rows_from_task(task: tuple[str, pd.DataFrame, str, int]) -> list[dict]:
    device_name, device_df, source_name, window_size = task
    return _build_rows_for_device(device_name, device_df, source_name, window_size)


# Find all PCAP files recursively under the selected root.
def list_pcap_files(root_path: Path) -> list[Path]:
    return sorted(root_path.rglob("*.pcapng"))


# Group PCAP files by first-level folder relative to ROOT_PATH.
def group_pcaps_by_subfolder(root_path: Path, pcap_files: list[Path]) -> dict[str, list[Path]]:
    groups: dict[str, list[Path]] = defaultdict(list)
    for pcap_file in pcap_files:
        rel_file = pcap_file.relative_to(root_path)
        group_name = rel_file.parts[0] if len(rel_file.parts) > 1 else "root"
        groups[group_name].append(pcap_file)
    return dict(groups)


# Build output filename for one subfolder.
def build_output_path(output_dir: Path, subfolder_name: str, window_size: int) -> Path:
    safe_name = subfolder_name.replace("/", "__")
    return output_dir / f"dataset_{safe_name}_{window_size}s.csv"


# Read address-to-device mappings for one PCAP.
def load_device_mappings(mapping_file: Path) -> tuple[dict, dict]:
    if not mapping_file.exists():
        return {}, {}

    try:
        mapping_df = pd.read_csv(mapping_file, dtype=str)
    except Exception:
        return {}, {}

    if "address" not in mapping_df.columns:
        return {}, {}

    work_df = mapping_df.copy()
    work_df["address"] = work_df["address"].fillna("").astype(str)
    work_df["device_name"] = work_df.get("device_name", "Unknown").fillna("Unknown").astype(str)
    work_df["device_type"] = work_df.get("device_type", "Unknown").fillna("Unknown").astype(str)

    name_map: dict[str, str] = {}
    type_map: dict[str, str] = {}
    for address, dev_name, dev_type in work_df[["address", "device_name", "device_type"]].itertuples(index=False, name=None):
        if address:
            name_map[address] = dev_name or "Unknown"
            type_map[address] = dev_type or "Unknown"

    return name_map, type_map


# Extract packet fields from one PCAP using tshark.
def extract_packet_dataframe(pcap_file: Path) -> pd.DataFrame:
    command = [TSHARK_PATH, "-n", "-r", str(pcap_file), "-T", "fields"]
    for field in TSHARK_FIELDS:
        command.extend(["-e", field])
    command.extend([
        "-E", "header=y",
        "-E", "separator=,",
        "-E", "quote=d",
        "-E", "occurrence=f",
    ])

    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        packet_df = pd.read_csv(StringIO(result.stdout))
    except Exception:
        return pd.DataFrame()

    if packet_df.empty:
        return packet_df

    packet_df = packet_df.rename(columns=COLUMN_RENAME_MAP)
    packet_df["Time"] = pd.to_datetime(packet_df["Time"], errors="coerce")
    packet_df = packet_df.dropna(subset=["Time"]).copy()

    for col in ["Length", "Packet Number"]:
        if col in packet_df.columns:
            packet_df[col] = pd.to_numeric(packet_df[col], errors="coerce")

    packet_df["Delta Time"] = packet_df["Time"].diff().dt.total_seconds().fillna(0)
    return packet_df


# Add device labels from mapping for source and destination addresses.
def add_groundtruth_columns(packet_df: pd.DataFrame, mapping_file: Path) -> pd.DataFrame:
    if packet_df.empty:
        return packet_df

    name_map, type_map = load_device_mappings(mapping_file)
    df = packet_df.copy()

    src16 = df.get("Source IEEE", pd.Series(index=df.index, dtype=object)).replace("", pd.NA)
    src64 = df.get("Source IEEE 64-bit", pd.Series(index=df.index, dtype=object)).replace("", pd.NA)
    dst16 = df.get("Destination IEEE", pd.Series(index=df.index, dtype=object)).replace("", pd.NA)
    dst64 = df.get("Destination IEEE 64-bit", pd.Series(index=df.index, dtype=object)).replace("", pd.NA)

    df["Source Address"] = src16.fillna(src64).fillna("")
    df["Destination Address"] = dst16.fillna(dst64).fillna("")

    df["Device Name"] = df["Source Address"].map(name_map).fillna("Unknown")
    df["Device Type"] = df["Source Address"].map(type_map).fillna("Unknown")
    df["Device Name Destination"] = df["Destination Address"].map(name_map).fillna("Unknown")
    df["Device Type Destination"] = df["Destination Address"].map(type_map).fillna("Unknown")

    return df.sort_values(["Device Name", "Time"]).reset_index(drop=True)


# Compute mean absolute deviation.
def calculate_mad(series: pd.Series) -> float:
    return (series - series.mean()).abs().mean()


# Compute statistical features for a numeric series.
def compute_stats(series: pd.Series, base_name: str) -> dict:
    return {
        f"Average {base_name}": series.mean(),
        f"{base_name} Standard Variance": series.std(),
        f"Maximum {base_name}": series.max(),
        f"Minimum {base_name}": series.min(),
        f"Median {base_name}": series.median(),
        f"Kurtosis {base_name}": series.kurtosis(),
        f"Skew {base_name}": series.skew(),
        f"Mad {base_name}": calculate_mad(series),
    }


# Extract one feature vector from one time window.
def extract_window_features(window_df: pd.DataFrame) -> dict | None:
    if len(window_df) < 2:
        return None

    lengths = window_df["Length"].dropna()
    iat = window_df["Time"].diff().dt.total_seconds().dropna()
    if len(lengths) < 2 or iat.empty:
        return None

    features = compute_stats(lengths, "Packet Length")
    features["Length"] = len(window_df)
    features.update(compute_stats(iat, "IAT"))
    return features


# Prefix feature names (UP, DOWN, TOT).
def with_prefix(features: dict | None, prefix: str) -> dict:
    if not features:
        return {}
    return {f"{prefix}_{key}": value for key, value in features.items()}


# Build all sliding-window rows for one device.
def _build_rows_for_device(device_name: str, device_df: pd.DataFrame, source_name: str, window_size: int) -> list[dict]:
    device_df = device_df.sort_values("Time").reset_index(drop=True)
    if len(device_df) < 2:
        return []

    rows: list[dict] = []
    step_ns = 1_000_000_000
    window_ns = window_size * step_ns
    time_ns = device_df["Time"].astype("int64").to_numpy()

    current_ns = int(time_ns[0])
    end_ns = int(time_ns[-1])
    left = 0
    right = 0
    row_count = len(device_df)

    while current_ns < end_ns:
        while left < row_count and int(time_ns[left]) < current_ns:
            left += 1

        window_end_ns = current_ns + window_ns
        while right < row_count and int(time_ns[right]) < window_end_ns:
            right += 1

        if right - left >= 2:
            window_df = device_df.iloc[left:right]
            total_features = extract_window_features(window_df)
            if total_features:
                up_mask = window_df["Device Name Destination"] == device_name
                up_features = extract_window_features(window_df[up_mask]) if up_mask.sum() >= 2 else None

                row = {
                    "Device Name": device_name,
                    "Device Type": window_df.iloc[0]["Device Type"],
                    "File Name": source_name,
                }
                row.update(with_prefix(total_features, "DOWN"))
                row.update(with_prefix(total_features, "TOT"))
                row.update(with_prefix(up_features, "UP"))
                rows.append(row)

        current_ns += step_ns

    return rows


# Build rows for one device using a subset of window start times.
def _build_rows_for_device_chunk(task: tuple[str, pd.DataFrame, str, int, list[int]]) -> list[dict]:
    device_name, device_df, source_name, window_size, start_ns_values = task
    if len(device_df) < 2 or not start_ns_values:
        return []

    device_df = device_df.sort_values("Time").reset_index(drop=True)
    time_ns = device_df["Time"].astype("int64").to_numpy()
    window_ns = window_size * 1_000_000_000
    rows: list[dict] = []

    for current_ns in start_ns_values:
        left = int(np.searchsorted(time_ns, current_ns, side="left"))
        right = int(np.searchsorted(time_ns, current_ns + window_ns, side="left"))
        if right - left < 2:
            continue

        window_df = device_df.iloc[left:right]
        total_features = extract_window_features(window_df)
        if not total_features:
            continue

        up_mask = window_df["Device Name Destination"] == device_name
        up_features = extract_window_features(window_df[up_mask]) if up_mask.sum() >= 2 else None

        row = {
            "Device Name": device_name,
            "Device Type": window_df.iloc[0]["Device Type"],
            "File Name": source_name,
        }
        row.update(with_prefix(total_features, "DOWN"))
        row.update(with_prefix(total_features, "TOT"))
        row.update(with_prefix(up_features, "UP"))
        rows.append(row)

    return rows


# Generate sliding-window features for one capture.
def create_feature_sequences(
    df: pd.DataFrame,
    source_name: str,
    window_size: int,
    device_workers: int = 1,
) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    df = df.copy()
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df["Length"] = pd.to_numeric(df["Length"], errors="coerce")
    df = df.dropna(subset=["Time"]).sort_values(["Device Name", "Time"])

    device_groups = list(df.groupby("Device Name", sort=False))
    if not device_groups:
        return pd.DataFrame()

    rows: list[dict] = []
    local_workers = max(1, min(device_workers, len(device_groups)))

    if local_workers > 1 and _step3_can_use_process_pool():
        tasks = [(device_name, device_df, source_name, window_size) for device_name, device_df in device_groups]
        executor_kwargs = {"mp_context": mp.get_context("fork")}
        try:
            with ProcessPoolExecutor(max_workers=local_workers, **executor_kwargs) as executor:
                for chunk in executor.map(_build_rows_from_task, tasks, chunksize=2):
                    rows.extend(chunk)
        except Exception:
            for device_name, device_df in device_groups:
                rows.extend(_build_rows_for_device(device_name, device_df, source_name, window_size))
    else:
        for device_name, device_df in device_groups:
            rows.extend(_build_rows_for_device(device_name, device_df, source_name, window_size))

    return pd.DataFrame(rows)

# Process one PCAP end-to-end.
def process_single_pcap(
    pcap_file: Path,
    window_size: int,
    fill_value: int,
    progress_tracker: NotebookProgressTracker,
    device_workers: int = 1,
) -> pd.DataFrame:
    file_name = pcap_file.name
    mapping_file = pcap_file.with_name(f"{pcap_file.stem}_mapping.csv")

    progress_tracker.update_step(file_name, 1, "Extract packets")
    packet_df = extract_packet_dataframe(pcap_file)
    if packet_df.empty:
        progress_tracker.complete(file_name)
        return packet_df

    progress_tracker.update_step(file_name, 2, "Map labels")
    labeled_df = add_groundtruth_columns(packet_df, mapping_file)

    progress_tracker.update_step(file_name, 3, "Build features")
    sequence_df = create_feature_sequences(
        labeled_df,
        pcap_file.name,
        window_size,
        device_workers=device_workers,
    )
    if sequence_df.empty:
        progress_tracker.complete(file_name)
        return sequence_df

    progress_tracker.update_step(file_name, 4, "Filter and fill")
    sequence_df = sequence_df[sequence_df["Device Name"] != "Unknown"].copy()
    sequence_df = sequence_df.fillna(fill_value)

    progress_tracker.complete(file_name)
    return sequence_df


# Run the full pipeline and save one CSV per subfolder.
def run_pipeline_by_subfolder(
    root_path: Path = ROOT_PATH,
    window_size: int = WINDOW_SIZE,
    fill_value: int = FILL_NA_VALUE,
    max_workers: int = CPU_COUNT,
    output_dir: Path = OUTPUT_DIR,
) -> pd.DataFrame:

    pcap_files = list_pcap_files(root_path)
    if not pcap_files:
        print("No .pcapng files found")
        return pd.DataFrame()

    grouped_pcaps = group_pcaps_by_subfolder(root_path, pcap_files)
    total_files = sum(len(files) for files in grouped_pcaps.values())

    # Define workers for both file-level and by-device Step 3 parallelism.
    file_workers = max(1, min(total_files, max_workers // 2 if max_workers > 1 else 1))
    device_workers = max(1, min(CPU_COUNT // 2, max_workers // file_workers))

    frames_by_subfolder: dict[str, list[pd.DataFrame]] = defaultdict(list)
    progress_tracker = NotebookProgressTracker(total_steps=4)

    with ThreadPoolExecutor(max_workers=file_workers) as executor:
        futures = {}
        for subfolder_name, files in grouped_pcaps.items():
            for pcap_file in files:
                future = executor.submit(
                    process_single_pcap,
                    pcap_file,
                    window_size,
                    fill_value,
                    progress_tracker,
                    device_workers,
                )
                futures[future] = (subfolder_name, pcap_file)

        for future in as_completed(futures):
            subfolder_name, done_pcap_file = futures[future]
            try:
                df = future.result()
            except Exception:
                progress_tracker.complete(done_pcap_file.name, fail=True)
                continue

            if not df.empty:
                frames_by_subfolder[subfolder_name].append(df)

    if not frames_by_subfolder:
        print("No final dataset generated")
        return pd.DataFrame()

    output_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []

    print("\nSaving output files...", end="")

    for subfolder_name in sorted(frames_by_subfolder.keys()):
        subfolder_df = pd.concat(frames_by_subfolder[subfolder_name], ignore_index=True)
        output_path = build_output_path(output_dir, subfolder_name, window_size)
        subfolder_df.to_csv(output_path, index=False, encoding="utf-8")

        source_file_count = subfolder_df["File Name"].nunique() if "File Name" in subfolder_df else 0
        summary_rows.append({
            "Subfolder": subfolder_name,
            "Output File": str(output_path),
            "Rows": len(subfolder_df),
            "Columns": len(subfolder_df.columns),
            "Source Files": source_file_count,
        })

    return pd.DataFrame(summary_rows)

if _step3_can_use_process_pool():
    print(f"Using up to {CPU_COUNT} workers for parallelization (ProcessPool)")
else:
    print(f"OS does not support ProcessPool, limited parallelization capabilities.\nExpect longer processing time, in particular for Step 3.")

Using up to 8 workers for parallelization (ProcessPool)


### Code execution

In [3]:
# Process all PCAP files and save one CSV for each subfolder.
output_summary_df = run_pipeline_by_subfolder()

# Print a compact summary of generated output files.
if output_summary_df.empty:
    print("No output files generated")
else:
    print("\rProcessing complete. Summary:")
    print(output_summary_df.to_string(index=False))

MatterCapture1.pcapng - Completed
MatterCapture2.pcapng - Completed
MatterCapture3.pcapng - Completed
ZigbeeCapture4.pcapng - Completed
ZigbeeCapture5.pcapng - Completed
ZigbeeCapture6.pcapng - Completed

Processing complete. Summary:
   Subfolder                         Output File   Rows  Columns  Source Files
MatterThread outputs/dataset_MatterThread_5s.csv 617021       54             3
      Zigbee       outputs/dataset_Zigbee_5s.csv 256927       54             3
